# Actividad 4. Benchmark de modelos preentrenados de Hugging Face

Comparación reproducible de tres modelos para análisis de sentimiento en español sobre 5,000 reseñas de prueba del Multilingual Amazon Reviews Corpus. El objetivo es contrastar desempeño predictivo, latencia y consumo de recursos en Google Colab con GPU.

In [ ]:
!nvidia-smi
!pip -q install -U transformers datasets huggingface_hub accelerate scikit-learn matplotlib

In [ ]:
import gc, time, random, platform
import numpy as np, pandas as pd, torch, transformers, datasets, sklearn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=0 if torch.cuda.is_available() else -1
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Dataset

Se usa `SetFit/amazon_reviews_multi_es`, partición `test`. Las etiquetas 0-4 representan 1-5 estrellas y se convierten a `NEG` (1-2), `NEU` (3) y `POS` (4-5).

In [ ]:
ds=load_dataset('SetFit/amazon_reviews_multi_es', split='test')
df=ds.to_pandas().reset_index(drop=True)
def to_sentiment(label):
    stars=int(label)+1
    return 'NEG' if stars<=2 else ('NEU' if stars==3 else 'POS')
df['id']=np.arange(len(df)); df['sentiment']=df['label'].apply(to_sentiment)
evaluation_df=df[['id','text','label','sentiment']].copy()
print(evaluation_df['sentiment'].value_counts())

## Modelos

- `nlptown/bert-base-multilingual-uncased-sentiment`: dominio de reseñas de productos.
- `pysentimiento/robertuito-sentiment-analysis`: texto social en español.
- `cardiffnlp/twitter-xlm-roberta-base-sentiment`: Twitter multilingüe.

In [ ]:
MODELS={
 'NLP Town mBERT':'nlptown/bert-base-multilingual-uncased-sentiment',
 'RoBERTuito':'pysentimiento/robertuito-sentiment-analysis',
 'XLM-R Twitter':'cardiffnlp/twitter-xlm-roberta-base-sentiment'
}
LABELS=['NEG','NEU','POS']
def norm(name,label):
    x=str(label).lower().strip()
    if name=='NLP Town mBERT':
        if x.startswith(('1','2')): return 'NEG'
        if x.startswith('3'): return 'NEU'
        if x.startswith(('4','5')): return 'POS'
    m={'negative':'NEG','neg':'NEG','neutral':'NEU','neu':'NEU','positive':'POS','pos':'POS'}
    if x in m:return m[x]
    raise ValueError((name,label))

## Evaluación

La inferencia usa batch size 32 y máximo 128 tokens. La latencia se mide después de warm-up y sincronización CUDA. La carga del checkpoint se excluye del tiempo principal de inferencia.

In [ ]:
def evaluate(name, model_id, texts, y_true):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter()
    tok=AutoTokenizer.from_pretrained(model_id)
    model=AutoModelForSequenceClassification.from_pretrained(model_id)
    load_s=time.perf_counter()-t0
    nparams=sum(p.numel() for p in model.parameters())
    clf=pipeline('text-classification', model=model, tokenizer=tok, device=DEVICE)
    _=clf(texts[:32], batch_size=32, truncation=True, max_length=128)
    if torch.cuda.is_available(): torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter(); out=clf(texts,batch_size=32,truncation=True,max_length=128)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0
    yp=[norm(name,o['label']) for o in out]
    p,r,f,_=precision_recall_fscore_support(y_true,yp,labels=LABELS,average='macro',zero_division=0)
    result={'Modelo':name,'Accuracy':accuracy_score(y_true,yp),'Precision_macro':p,'Recall_macro':r,
            'F1_macro':f,'Tiempo_carga_s':load_s,'Tiempo_inferencia_s':elapsed,
            'Latencia_media_ms':elapsed/len(texts)*1000,'Throughput_reseñas_s':len(texts)/elapsed,
            'Parametros_M':nparams/1e6,'VRAM_pico_MB':torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else np.nan}
    pred=pd.DataFrame({'id':evaluation_df.id,'texto':texts,'real':y_true,'prediccion':yp,'confianza':[o['score'] for o in out],'modelo':name})
    report=classification_report(y_true,yp,labels=LABELS,output_dict=True,zero_division=0)
    cm=confusion_matrix(y_true,yp,labels=LABELS)
    del clf,model,tok; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result,pred,report,cm

In [ ]:
texts=evaluation_df.text.astype(str).tolist(); y_true=evaluation_df.sentiment.tolist()
metrics=[]; predictions=[]; reports={}; cms={}
for name,mid in MODELS.items():
    res,pr,rep,cm=evaluate(name,mid,texts,y_true)
    metrics.append(res); predictions.append(pr); reports[name]=rep; cms[name]=cm
metrics_df=pd.DataFrame(metrics).sort_values('F1_macro',ascending=False).reset_index(drop=True)
predictions_df=pd.concat(predictions,ignore_index=True)
display(metrics_df)

## Resultados de la ejecución documentada

En la corrida final sobre Tesla T4, NLP Town mBERT obtuvo `accuracy=0.7650` y `F1_macro=0.7153`; RoBERTuito `0.6836` y `0.6191`; XLM-R Twitter `0.6754` y `0.5882`. RoBERTuito fue el más rápido, mientras que NLP Town presentó el mejor balance general. La clase neutral fue la más difícil para los tres modelos.

In [ ]:
from pathlib import Path
Path('data').mkdir(exist_ok=True); Path('results').mkdir(exist_ok=True)
evaluation_df.to_csv('data/dataset_evaluacion.csv',index=False)
metrics_df.to_csv('results/metricas_modelos.csv',index=False)
predictions_df.to_csv('results/predicciones_modelos.csv',index=False)